## Phi-4 RAG နှင့် Azure AI ရှာဖွေခြင်း

ဤ notebook တွင် Phi-4-mini နှင့် Phi-4-multimodal ကို Retrieval Augmented Generation (RAG) အတွက် Azure AI ရှာဖွေမှုအတူအသုံးပြုနည်းကို ပြသသည်။ ၎င်းသည် single-modality (စာသားသာ) နှင့် multi-modality (စာသားနှင့်ပုံ) ဇာတ်လမ်းများနှစ်မျိုးလုံးကို ပါဝင်သည်။

**လိုအပ်ချက်များ:**
*   Azure AI ရှာဖွေ vector အညွှန်း (တစ်ခုဖန်တီးရန် [ဤညွှန်ကြားချက်များ](https://learn.microsoft.com/azure/search/search-get-started-portal-import-vectors?tabs=sample-data-storage%2Cmodel-aoai%2Cconnect-data-storage)ကိုလိုက်နာပါ)
*   Microsoft Foundry တွင် Phi-4-mini သို့မဟုတ် Phi-4-multimodal endpoints တပ်ဆင်ထားခြင်း


In [ ]:
! pip install azure-ai-inference azure-search-documents

### Text-Only RAG with Phi-4-mini

ဤအစိတ်အပိုင်းသည် Phi-4-mini ကို RAG အတွက် စကားပြောပြီးဆုံးမှုမော်ဒယ်အဖြစ် အသုံးပြုနည်းကို တစ်လုံးတည်းစာသားအဖြစ် အသုံးပြုသည့်နည်းကို ပြသသည်။ ၎င်းမှာ Microsoft Foundry Inference နှင့် Azure AI Search ကို ချိတ်ဆက်ပြီး သက်ဆိုင်ရာစာတမ်းများကို ရယူကာ ရရှိထားသောအကြောင်းအရာများကို အသုံးပြု၍ ဖြေကြားချက်တစ်ခုကို ဖန်တီးခြင်း ပါဝင်သည်။


In [ ]:
# Step 1: Connect to Microsoft Foundry Inference & Azure AI Search
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import UserMessage
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

chat_client = ChatCompletionsClient(
    endpoint=os.environ["AZURE_INFERENCE_ENDPOINT"], # Phi-4-mini endpoint
    credential=AzureKeyCredential(os.environ["AZURE_INFERENCE_CREDENTIAL"]),
)

search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name=os.environ["AZURE_SEARCH_INDEX"],
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

# Step 2: Retrieve relevant documents from Azure AI Search
def retrieve_documents(query: str, top: int = 10):
    vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=top, fields="text_vector")
    results = search_client.search(search_text=query,vector_queries=[vector_query],select=["content"],top=top)
    return [doc["content"] for doc in results]

# Step 3: Generate answer using RAG!
def generate_rag_response(query: str):
    docs = retrieve_documents(query)
    context = "\n---\n".join(docs)
    prompt = f"""You are a helpful assistant. Use only the following context to answer the question. If the answer isn't in the context, say 'I don't know'.
    Context: {context} Question: {query} Answer:"""
    response = chat_client.complete(messages=[UserMessage(content=prompt)])
    return response.choices[0].message.content

# Example usage
user_query = "What is the capital of France?"
answer = generate_rag_response(user_query)
print(f"Q: {user_query}\nA: {answer}")


### Phi-4-multimodal နှင့် Multi-Modal RAG

ဤအပိုဒ်တွင် Phi-4-multimodal ကို စကားပြောပြီးစီးမှုမော်ဒယ်အဖြစ် RAG အတွက်အသုံးပြုပုံကို ဖော်ပြထားသည်။ စာသားနှင့်ပုံရိပ် input နှစ်ခုလုံးကို ပေါင်းစပ်အသုံးပြုသည်။ Azure AI Inference နှင့် Azure AI Search တွင် ချိတ်ဆက်ခြင်း၊ သက်ဆိုင်ရာစာရွက်စာတမ်းများ ရယူခြင်းနှင့် မော်ဒယ်စုံဖော်ပြချက်တစ်ခု ထုတ်ပေးခြင်းတို့ကို ဖော်ပြသည်။

**Note:** သင်မှာ Azure AI Search index တွင် `text_vector` နှင့် `image_vector` field နှစ်ခုလုံး ရှိပါက multi-vector query ကိုလည်း ပြုလုပ်နိုင်ပါသည်။


In [ ]:
# Step 1: Connect to Azure AI Inference & Azure AI Search
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery


chat_client = ChatCompletionsClient(
    endpoint=os.environ["AZURE_INFERENCE_ENDPOINT"], #Phi-4-multimodal serverless endpoint
    credential=AzureKeyCredential(os.environ["AZURE_INFERENCE_CREDENTIAL"]),
)

search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name=os.environ["AZURE_SEARCH_INDEX"],
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

# Step 2: Retrieve relevant documents from Azure AI Search
def retrieve_documents(query: str, top: int = 3):
    vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=top, fields="text_vector")
    results = search_client.search(search_text=query, vector_queries=[vector_query], select=["content"], top=top)    
    return [doc["content"] for doc in results]

## Example for Muli-modal search if you have a text_vector AND image_vector field in your vector_index
## NOTE, image vectorization is in preview at the time of writing this code, please use azure-search-documents pypi version >11.6.0b6 
# def retrieve_documents_multimodal(query: str, image_url: str, top: int = 3):
#     text_vector_query = VectorizableTextQuery(
#         text=query,
#         k_nearest_neighbors=top,
#         fields="text_vector",
#         weight=0.5  # Adjust weight as needed
#     )
#     image_vector_query = VectorizableImageUrlQuery(
#         url=image_url,
#         k_nearest_neighbors=top,
#         fields="image_vector",
#         weight=0.5  # Adjust weight as needed
#     )

#     results = search_client.search(
#         search_text=query,  
#         vector_queries=[text_vector_query, image_vector_query],
#         select=["content"],
#         top=top
#     )
#     return [doc["content"] for doc in results]


# Step 3: Generate a multimodal RAG-based answer using retrieved text and an image input
def generate_multimodal_rag_response(query: str, image_url: str):
    # Retrieve text context from search
    docs = retrieve_documents(query)
    context = "\n---\n".join(docs)

    # Build a prompt that combines the retrieved context with the user query
    prompt = f"""You are a helpful assistant. Use only the following context to answer the question. If the answer isn't in the context, say 'I don't know'.
    Context: {context} Question: {query} Answer:"""
    # Create a chat request that includes both text and image input
    response = chat_client.complete(
        messages=[
            SystemMessage(content="You are a helpful assistant that can process both text and images."),
            UserMessage(
                content=[
                    TextContentItem(text=prompt),
                    ImageContentItem(image_url=ImageUrl(url=image_url, detail=ImageDetailLevel.HIGH)),
                ]
            ),
        ]
    )
    return response.choices[0].message.content

# Example usage
user_query = "Can you search for similar items to this shoe?"
sample_image_url = "https://images.unsplash.com/photo-1542291026-7eec264c27ff?q=80&w=1770&auto=format&fit=crop&ixlib=rb-4.0.3"
answer = generate_multimodal_rag_response(user_query, sample_image_url)
print(f"Q: {user_query}\nA: {answer}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ကြေညာချက်**:  
ဤစာရွက်စာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) ဖြင့် ဘာသာပြန်ထားသည်။ ကျနော်တို့သည် တိကျမှုအတွက် ကြိုးစားသော်လည်း၊ အလိုအလျောက် ဘာသာပြန်ချက်များတွင် အမှားများ သို့မဟုတ် တိကျမှုမရှိနိုင်ခြင်းဖြစ်နိုင်ပါသည်။ မူရင်းစာရွက်စာတမ်းကို ဆက်စပ်ဘာသာစကားဖြင့်ပင် အဓိက အရင်းအမြစ်အဖြစ် မှတ်ယူသင့်ပါသည်။ အရေးကြီးသော သတင်းအချက်အလက်များအတွက် လူမြောက်ပညာရှင်တို့၏ ဘာသာပြန်ခြင်းကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက် အသုံးပြုမှုကြောင့် ဖြစ်ပေါ်လာသော နားလည်မှု အညစ်အကြေးများအတွက် ကျနော်တို့ တာဝန်မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
